### Checking null count in 'orders' table and then creating silver table for the same

In [0]:
#load bronze 'orders' table
from pyspark.sql.functions import col, when, sum

orders_df = spark.table('ecommerce_project.bronze.orders')

display(orders_df.limit(10))
orders_df.printSchema()


In [0]:
# check null values
null_cnt = orders_df.select([
    sum(when(col(column).isNull(), 1).otherwise(0)).alias(column)
    for column in orders_df.columns
])

display(null_cnt)

### As there are no null values in this table, hence no null handling is required.
### Lets create Silver table of 'orders'

In [0]:
# creating Silver 'Orders' table
# cogs_usd : cost of goods sold in usd

from pyspark.sql.functions import to_date, round

silver_order_df = (
    orders_df
    .dropDuplicates(['order_id'])
    .withColumn('order_date', to_date(col('created_at')))
    .withColumn(
        "gross_profit_usd", round(col('price_usd') - col('cogs_usd'), 2)
    )
)

In [0]:
(
    silver_order_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('ecommerce_project.silver.orders')
)

In [0]:
spark.table('ecommerce_project.silver.orders').count()

### Lets do for next table 'order_items'

In [0]:
#Load bronze table
order_items_df = spark.table("ecommerce_project.bronze.order_items")

display(order_items_df.limit(10))


#check null values
null_cnt = order_items_df.select([
    sum(when(col(column).isNull(), 1).otherwise(0)).alias(column)
    for column in order_items_df.columns
])

display(null_cnt)

### As there are no null values in this table, hence no null handling is required.
### Lets create Silver table of 'order_items'



In [0]:
# creating Silver 'order_items' table
#Add the item date, gross profit and profit margin

silver_order_items_df = (
    order_items_df
    .dropDuplicates(['order_item_id'])
    .withColumn('order_item_date', to_date(col('created_at')))
    .withColumn(
        "gross_profit_usd", round(col('price_usd') - col('cogs_usd'), 2)
    )
    .withColumn(
        "profit_margin_percentage", round(((col('price_usd') / col('cogs_usd')) / col('price_usd')) * 100,
    )
)
)

In [0]:
(
    silver_order_items_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('ecommerce_project.silver.order_items')
)

In [0]:
spark.table("ecommerce_project.silver.order_items").count()

### Lets do for next table 'order_item_refunds'

In [0]:
#Load bronze table
refunds_df = spark.table(
    "ecommerce_project.bronze.order_item_refunds"
)

display(refunds_df.limit(10))


#check null values
refund_null_counts = refunds_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in refunds_df.columns
])

display(refund_null_counts)

### Creating Silver table of: 'order_item_refunds' 

In [0]:
silver_refunds_df = (
    refunds_df
    .dropDuplicates(["order_item_refund_id"])
    .withColumn("refund_date", to_date(col("created_at")))
)

In [0]:
(
    silver_refunds_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('ecommerce_project.silver.order_item_refunds')
    )

In [0]:
spark.table("ecommerce_project.silver.order_item_refunds").count()

### Lets do for next table 'products'

In [0]:
#Load bronze table
products_df = spark.table(
    "ecommerce_project.bronze.products"
)

display(products_df.limit(10))


#check null values
products_null_counts = products_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in products_df.columns
])

display(products_null_counts)

In [0]:
# Clean and transform
silver_products_df = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_created_date",
        to_date(col("created_at"))
    )
)

In [0]:
#write silver table
(
    silver_products_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('ecommerce_project.silver.products')
)

In [0]:
spark.table("ecommerce_project.silver.products").count()

### Lets do for next table 'website_sessions'

In [0]:
#Load bronze table
sessions_df = spark.table(
    "ecommerce_project.bronze.website_sessions"
)

display(sessions_df.limit(10))


In [0]:
#check null values
sessions_null_counts = sessions_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in sessions_df.columns
])

display(sessions_null_counts)


In [0]:
#check duplicate primary keys
duplicate_sessions = (
    sessions_df
    .groupBy("website_session_id")
    .count()
    .filter(col("count") > 1)
)
print(duplicate_sessions.count())


In [0]:
# Clean and transform
silver_sessions_df = (
    sessions_df
    .dropDuplicates(["website_session_id"])
    .withColumn(
        "session_date",
        to_date(col("created_at"))
    )
    .withColumn(
        "traffic_channel",
        when(col("utm_source").isNotNull(), col("utm_source"))
        .when(col("http_referer").isNotNull(), "organic")
        .otherwise("direct")
    )
)



In [0]:
# Write Silver table
(
    silver_sessions_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_project.silver.website_sessions")
)

In [0]:
spark.table("ecommerce_project.silver.website_sessions").count()

### Lets do for tazble 'website_pageviews'

In [0]:
# Load Bronze table
pageviews_df = spark.table(
    "ecommerce_project.bronze.website_pageviews"
)

display(pageviews_df.limit(10))


In [0]:
# Check null values
pageviews_null_counts = pageviews_df.select([
    sum(
        when(col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in pageviews_df.columns
])

display(pageviews_null_counts)

In [0]:
# Check duplicate primary keys
duplicate_pageviews = (
    pageviews_df
    .groupBy("website_pageview_id")
    .count()
    .filter(col("count") > 1)
)

print(
    "Duplicate website pageview IDs:",
    duplicate_pageviews.count()
)

In [0]:
# Clean and transform
from pyspark.sql.functions import lower, trim
silver_pageviews_df = (
    pageviews_df
    .dropDuplicates(["website_pageview_id"])
    .withColumn(
        "pageview_date",
        to_date(col("created_at"))
    )
    .withColumn(
        "pageview_url",
        lower(trim(col("pageview_url")))
    )
)

In [0]:
# Write Silver table
(
    silver_pageviews_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_project.silver.website_pageviews")
)


In [0]:
spark.table("ecommerce_project.silver.website_pageviews").count()

In [0]:
display(silver_pageviews_df.limit(10))